# ✏️ TOONIFY — Sketch LoRA Model Training Notebook

Train a custom Stable Diffusion 1.5 LoRA model using your **400-sketch dataset** (`20_abhi_sketch_style person`).

- **Base Model**: `runwayml/stable-diffusion-v1-5`
- **Trigger Word**: `abhi_sketch_style`
- **Dataset**: PNG images + matching `.txt` caption files
- **Output**: `sketch_lora.safetensors`

## Step 1: Install Dependencies
Run this cell on Google Colab (with T4 GPU enabled).

In [ ]:
!nvidia-smi
!pip install -q diffusers==0.27.2 transformers accelerate bitsandbytes safetensors torchvision xformers

## Step 2: Mount Google Drive & Prepare Dataset
Checks for either `sketch_dataset.zip` or the `sketch_dataset` folder in Google Drive.

In [ ]:
import os
import zipfile
import shutil
from google.colab import drive

if os.path.exists('/content/drive'):
    print('Google Drive already mounted.')
else:
    drive.mount('/content/drive')

dataset_dir = '/content/dataset'
os.makedirs(dataset_dir, exist_ok=True)

drive_folder = '/content/drive/MyDrive/sketch_dataset'
zip_path_drive = '/content/drive/MyDrive/sketch_dataset.zip'
zip_path_local = '/content/sketch_dataset.zip'

if os.path.exists(drive_folder):
    print(f'✅ Found sketch_dataset folder directly in Google Drive!')
    for root, dirs, files in os.walk(drive_folder):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg', '.txt')):
                src = os.path.join(root, file)
                dst = os.path.join(dataset_dir, file)
                shutil.copy2(src, dst)
    print('Dataset copied from Google Drive folder successfully!')

elif os.path.exists(zip_path_drive) or os.path.exists(zip_path_local):
    zip_target = zip_path_drive if os.path.exists(zip_path_drive) else zip_path_local
    print(f'Unzipping dataset from {zip_target}...')
    with zipfile.ZipFile(zip_target, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print('Dataset extracted successfully!')
else:
    print('⚠️ Please ensure sketch_dataset folder or zip is in Google Drive.')

## Step 3: Verify Dataset Pairs (.png + .txt)

In [ ]:
import glob

image_files = sorted(glob.glob('/content/dataset/**/*.png', recursive=True) + glob.glob('/content/dataset/**/*.jpg', recursive=True))
print(f'Found {len(image_files)} image files.')

valid_pairs = []
for img in image_files:
    txt_file = os.path.splitext(img)[0] + '.txt'
    if os.path.exists(txt_file):
        with open(txt_file, 'r', encoding='utf-8') as f:
            caption = f.read().strip()
        valid_pairs.append((img, caption))

print(f'✅ Found {len(valid_pairs)} complete image + caption pairs.')
if valid_pairs:
    print('Sample image:', valid_pairs[0][0])
    print('Sample caption:', valid_pairs[0][1])

## Step 4: Train Sketch LoRA Model (Native Diffusers)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from diffusers.models.attention_processor import LoRAAttnProcessor
from tqdm.auto import tqdm

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
OUTPUT_DIR = '/content/output_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder='text_encoder').to('cuda', dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae').to('cuda', dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet').to('cuda', dtype=torch.float16)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

# Native Diffusers LoRA Attention Processors
lora_attn_procs = {}
for name in unet.attn_processors.keys():
    cross_attention_dim = None if name.endswith('attn1.processor') else unet.config.cross_attention_dim
    if name.startswith('mid_block'):
        hidden_size = unet.config.block_out_channels[-1]
    elif name.startswith('up_blocks'):
        block_id = int(name.split('.')[1])
        hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
    elif name.startswith('down_blocks'):
        block_id = int(name.split('.')[1])
        hidden_size = unet.config.block_out_channels[block_id]

    lora_attn_procs[name] = LoRAAttnProcessor(hidden_size=hidden_size, cross_attention_dim=cross_attention_dim)

unet.set_attn_processor(lora_attn_procs)
lora_layers = torch.nn.ModuleList(unet.attn_processors.values()).to('cuda')

class SketchDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        self.transform = transforms.Compose([
            transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_path, caption = self.pairs[idx]
        image = Image.open(img_path).convert('RGB')
        tensor = self.transform(image)
        tokens = tokenizer(
            caption,
            padding='max_length',
            max_length=tokenizer.model_max_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)
        return {'image': tensor, 'tokens': tokens}

dataset = SketchDataset(valid_pairs)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.AdamW(lora_layers.parameters(), lr=1e-4)
num_epochs = 5
print(f'🚀 Starting native Diffusers LoRA training for {num_epochs} epochs ({len(dataloader) * num_epochs} steps)...')

unet.train()
for epoch in range(num_epochs):
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for batch in progress_bar:
        images = batch['image'].to('cuda', dtype=torch.float16)
        tokens = batch['tokens'].to('cuda')

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * 0.18215
            encoder_hidden_states = text_encoder(tokens)[0]

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device='cuda').long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction='mean')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

unet.save_attn_procs(OUTPUT_DIR)
print(f'✅ Training complete! Saved LoRA model to {OUTPUT_DIR}')

## Step 5: Test Model Generation in Colab

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None
).to('cuda')

pipe.unet.load_attn_procs(OUTPUT_DIR)

prompt = 'portrait of a young woman, abhi_sketch_style, pencil sketch, detailed crosshatching, graphite art'
image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save('/content/test_sketch_output.png')
display(image)